In [5]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

In [6]:
import os
import json
from datetime import datetime, timezone

import requests
import boto3
from botocore.client import Config


OPEN_METEO_URL = "https://api.open-meteo.com/v1/forecast"


def fetch_current_and_past_hours(latitude: float, longitude: float, past_hours: int = 24) -> dict:
    # We treat Open-Meteo as an "observations feed": current + recent past hours.
    # We do NOT use forecast horizons as our "model".
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "timezone": "Europe/Athens",
        "current": ",".join([
            "temperature_2m",
            "relative_humidity_2m",
            "dew_point_2m",
            "apparent_temperature",
            "precipitation",
            "rain",
            "showers",
            "snowfall",
            "weather_code",
            "pressure_msl",
            "surface_pressure",
            "cloud_cover",
            "wind_speed_10m",
            "wind_direction_10m",
            "wind_gusts_10m",
            "is_day",
        ]),
        "hourly": ",".join([
            "temperature_2m",
            "relative_humidity_2m",
            "dew_point_2m",
            "apparent_temperature",
            "precipitation",
            "rain",
            "showers",
            "snowfall",
            "weather_code",
            "pressure_msl",
            "surface_pressure",
            "cloud_cover",
            "wind_speed_10m",
            "wind_direction_10m",
            "wind_gusts_10m",
            "is_day",
        ]),
        "past_hours": past_hours,
        # Keep the response tight: no multi-day forecasts
        "forecast_hours": 0,
    }

    resp = requests.get(OPEN_METEO_URL, params=params, timeout=30)
    resp.raise_for_status()
    return resp.json()
print(fetch_current_and_past_hours)


<function fetch_current_and_past_hours at 0x112e3ca60>


In [7]:
# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://api.open-meteo.com/v1/forecast"
params = {
	"latitude": 38.2444,
	"longitude": 21.7344,
	"current": ["temperature_2m", "relative_humidity_2m", "apparent_temperature", "is_day", "precipitation", "rain", "showers", "snowfall", "weather_code", "pressure_msl", "surface_pressure", "cloud_cover", "wind_speed_10m", "wind_direction_10m", "wind_gusts_10m"],
	"past_days": 1,
	"forecast_days": 3,
	"forecast_hours": 24,
	"past_hours": 24,
}
responses = openmeteo.weather_api(url, params=params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process current data. The order of variables needs to be the same as requested.
current = response.Current()
current_temperature_2m = current.Variables(0).Value()
current_relative_humidity_2m = current.Variables(1).Value()
current_apparent_temperature = current.Variables(2).Value()
current_is_day = current.Variables(3).Value()
current_precipitation = current.Variables(4).Value()
current_rain = current.Variables(5).Value()
current_showers = current.Variables(6).Value()
current_snowfall = current.Variables(7).Value()
current_weather_code = current.Variables(8).Value()
current_pressure_msl = current.Variables(9).Value()
current_surface_pressure = current.Variables(10).Value()
current_cloud_cover = current.Variables(11).Value()
current_wind_speed_10m = current.Variables(12).Value()
current_wind_direction_10m = current.Variables(13).Value()
current_wind_gusts_10m = current.Variables(14).Value()

print(f"\nCurrent time: {current.Time()}")
print(f"Current temperature_2m: {current_temperature_2m}")
print(f"Current relative_humidity_2m: {current_relative_humidity_2m}")
print(f"Current apparent_temperature: {current_apparent_temperature}")
print(f"Current is_day: {current_is_day}")
print(f"Current precipitation: {current_precipitation}")
print(f"Current rain: {current_rain}")
print(f"Current showers: {current_showers}")
print(f"Current snowfall: {current_snowfall}")
print(f"Current weather_code: {current_weather_code}")
print(f"Current pressure_msl: {current_pressure_msl}")
print(f"Current surface_pressure: {current_surface_pressure}")
print(f"Current cloud_cover: {current_cloud_cover}")
print(f"Current wind_speed_10m: {current_wind_speed_10m}")
print(f"Current wind_direction_10m: {current_wind_direction_10m}")
print(f"Current wind_gusts_10m: {current_wind_gusts_10m}")



Coordinates: 38.25°N 21.75°E
Elevation: 20.0 m asl
Timezone difference to GMT+0: 0s

Current time: 1770159600
Current temperature_2m: 11.39799976348877
Current relative_humidity_2m: 72.0
Current apparent_temperature: 9.499859809875488
Current is_day: 0.0
Current precipitation: 0.0
Current rain: 0.0
Current showers: 0.0
Current snowfall: 0.0
Current weather_code: 2.0
Current pressure_msl: 1018.7999877929688
Current surface_pressure: 1016.3572998046875
Current cloud_cover: 73.0
Current wind_speed_10m: 6.989935398101807
Current wind_direction_10m: 78.11136627197266
Current wind_gusts_10m: 13.679999351501465
